In [ ]:
from netCDF4 import Dataset
from matplotlib import pyplot as plt
import pandas as pd
import numpy as np
import datetime
import pickle
from scipy.interpolate import griddata
from mpl_toolkits.basemap import Basemap
import os
from sklearn.linear_model import LinearRegression
from matplotlib.ticker import FixedLocator, MultipleLocator, IndexLocator

In [ ]:
months = [
    "2015-07", "2015-08", "2015-09", "2015-10",
    "2016-07", "2016-08", "2016-09", "2016-10",
    "2017-07", "2017-08", "2017-09", "2017-10",
    "2018-07", "2018-08", "2018-09", "2018-10",
    "2019-07", "2019-08", "2019-09", "2019-10",
    "2020-07", "2020-08", "2020-09", "2020-10",
    "2021-07", "2021-08", "2021-09", "2021-10",
    "2022-07", "2022-08", "2022-09", "2022-10",
    "2023-07", "2023-08", "2023-09", "2023-10",
    "2024-07", "2024-08", "2024-09", "2024-10",
]

In [ ]:
months = [
    "2000-08", "2000-09",
    "2001-08", "2001-09",
    "2002-08", "2002-09",
    "2003-08", "2003-09",
    "2004-08", "2004-09",
    "2005-08", "2005-09",
    "2006-08", "2006-09",
    "2007-08", "2007-09",
    "2008-08", "2008-09",
    "2009-08", "2009-09",
    "2010-08", "2010-09",
    "2011-08", "2011-09", 
    "2012-08", "2012-09",
    "2013-08", "2013-09",
    "2014-08", "2014-09",
    "2015-08", "2015-09",
    "2016-08", "2016-09",
    "2017-08", "2017-09",
    "2018-08", "2018-09",
    "2019-08", "2019-09",
    "2020-08", "2020-09",
    "2021-08", "2021-09",
    "2022-08", "2022-09",
    "2023-08", "2023-09",
    "2024-08", "2024-09",
]

In [ ]:
months = [
    "2015-08", "2015-09",
    "2016-08", "2016-09",
    "2017-08", "2017-09",
    "2018-08", "2018-09",
    "2019-08", "2019-09",
    "2020-08", "2020-09",
    "2021-08", "2021-09",
    "2022-08", "2022-09",
    "2023-08", "2023-09",
    "2024-08", "2024-09",
]

In [ ]:
wind_speed_hourly_2d = []
wind_speed_monthly = []
for month in months:
    with open(f'/mnt/hippocamp/asavin/data/wind/wind_statistics_kara_n78_s73_w65_e85/{month}.pkl', 'rb') as file:
        data = pickle.load(file)

    wind_speed_hourly_2d.append(data['wind_speed_hourly_2d'].mean())
    wind_speed_monthly.append(data['wind_speed_monthly'])

In [ ]:
diff = [a - b for a, b in zip(wind_speed_hourly_2d, wind_speed_monthly)]

In [ ]:
# Позиции точек по оси X: 0, 1, 2, ...
pos = np.arange(len(months))

fig, ax = plt.subplots(figsize=(8, 4))

# Строим по числовым позициям
ax.plot(pos, wind_speed_hourly_2d, linestyle='-', label='Mean_2d', color='red')
ax.plot(pos, wind_speed_monthly, linestyle='-', label='Month_mean', color='blue')
# ax.plot(pos, diff, linestyle='-', label='Diff', color='green')

# Глобальные средние
mean_red = np.mean(wind_speed_hourly_2d)
mean_blue = np.mean(wind_speed_monthly)
# mean_green = np.mean(diff)

ax.axhline(y=mean_red, color='red', linestyle='--', linewidth=1.5, zorder=4)
ax.axhline(y=mean_blue, color='blue', linestyle='--', linewidth=1.5, zorder=4)
# ax.axhline(y=mean_green, color='green', linestyle='--', linewidth=1.5, zorder=4)

# Глобальные STD и бледная заливка на всю ширину графика
std_red = np.std(wind_speed_hourly_2d)
std_blue = np.std(wind_speed_monthly)
# std_green = np.std(diff)

# Горизонтальные полосы (axhspan рисует полосу по всей ширине по X)
span_red = ax.axhspan(mean_red - std_red, mean_red + std_red, facecolor='red', alpha=0.12, zorder=1)
span_blue = ax.axhspan(mean_blue - std_blue, mean_blue + std_blue, facecolor='blue', alpha=0.12, zorder=1)
# span_green = ax.axhspan(mean_green - std_green, mean_green + std_green, facecolor='green', alpha=0.12, zorder=1)
# Примечание: для зелёной можно просто использовать ax.axhspan(mean_green - std_green, mean_green + std_green, ...), как для других.
# Здесь показан эквивалентный способ с вычислением ymax; обычно достаточно одинакового вызова ax.axhspan(...).

# Индексы августов и подписи (годы)
august_idx = [i for i, d in enumerate(months) if d.endswith("-08")]
years = [d.split("-")[0] for d in months if d.endswith("-08")]

# 1) Минорные тики = вертикальные линии сетки между точками (на полуцелых)
grid_between_points = pos[::2][:-1] + 1.5
ax.xaxis.set_minor_locator(FixedLocator(grid_between_points))
# Сетка поярче: увеличиваем толщину и непрозрачность
ax.grid(which='minor', axis='x', linestyle='-', linewidth=1, alpha=0.95)

# 2) Мажорные тики = только подписи (без рисок), в центрах интервалов между вертикальными линиями
ax.set_xticks(august_idx)
ax.set_xticklabels(years, rotation=0)

# Скрываем риски у мажорных тиков (будет "без специальных отметок")
ax.tick_params(axis='x', which='major', length=0)

# Полностью скрываем минорные тики и их подписи (оставляем только линии сетки)
ax.tick_params(axis='x', which='minor', bottom=False, labelbottom=False)

# 3) Горизонтальные линии сетки на каждом целом значении по Y и делаем их поярче
ax.yaxis.set_major_locator(MultipleLocator(1))
ax.grid(which='major', axis='y', linestyle='-', linewidth=1, alpha=0.95)

# Аккуратно задаём пределы, чтобы сетка и подписи не "обрезались"
ax.set_xlim(-0.5, len(months) - 0.5)
ax.set_ylim(0, 9)

plt.xlabel('year')
plt.ylabel('wind speed, m/s')
# plt.legend()
plt.tight_layout()
plt.show()

In [ ]:
np.mean(wind_speed_hourly_2d), np.mean(wind_speed_monthly), np.mean(diff)

In [ ]:
np.std(wind_speed_hourly_2d), np.std(wind_speed_monthly), np.std(diff)

In [ ]:
lon_min_era5, lon_max_era5, lat_min_era5, lat_max_era5 = 35, 105, 66, 82
lat1, lat2 = (90-lat_max_era5)*4, (90-lat_min_era5)*4+1
lon1, lon2 = lon_min_era5*4, lon_max_era5*4+1

In [ ]:
month = '2020-09'
file = f'/mnt/hippocamp/DATA/ERA5/w10/era5_uv10m_{month}.nc'
data = Dataset(file, 'r')

longitude = np.array(data.variables['longitude'][lon1:lon2])
latitude = np.array(data.variables['latitude'][lat1:lat2])
lon_grid, lat_grid = np.meshgrid(longitude, latitude)
data.close()

In [ ]:
def drawing(u, v, t=None, lon = lon_grid, lat = lat_grid, save=False):
    fig = plt.figure(figsize=(24, 24), dpi=300)
    
    m = Basemap(width=1800000, height=1300000,
                resolution='l', projection='aea',
                lat_1=50, lat_2=55, lon_0=70, lat_0=74)
    m.drawcoastlines()
    m.fillcontinents(color='grey', lake_color='white')
    m.drawparallels(np.arange(-80., 90., 2.), labels=[False, True, True, False])
    m.drawmeridians(np.arange(-180., 180., 5.), labels=[True, True, False, True], latmax=90)
    m.drawmapboundary(fill_color='white')
    
    # Преобразование координат для карты
    x, y = m(lon, lat)
    
    # Вычисление скорости ветра
    speed = np.sqrt(u**2 + v**2)
    
    # Отображение скорости ветра
    speed_plot = m.pcolormesh(x, y, speed, shading='gouraud', cmap='jet', vmin=0, vmax=21)
    cbar=plt.colorbar(speed_plot, label='Wind speed (m/s)', shrink=0.30)
    # cbar.set_ticks([0,2,4,6,8,10,12])
    
    # Отображение направления ветра
    m.quiver(x[::4, ::4], y[::4, ::4], u[::4, ::4], v[::4, ::4], scale=30, scale_units='inches')
    plt.title(f'Wind speed {t}')
    
    plt.show()
    plt.close('all')

In [ ]:
month = '2024-09'
file = f'/mnt/hippocamp/DATA/ERA5/w10/era5_uv10m_{month}.nc'
data = Dataset(file, 'r')

tt = np.asarray(data.variables['valid_time'])
time = np.asarray([datetime.datetime(1970, 1, 1, 0, 0, 0) + datetime.timedelta(seconds=int(t)) for t in tt])
u10 = data.variables['u10'][:,lat1:lat2,lon1:lon2]
v10 = data.variables['v10'][:,lat1:lat2,lon1:lon2]
data.close()

In [ ]:
date = datetime.datetime(2024, 9, 8, 0, 0)
t = np.where(time == date)[0][0]
print(t)

In [ ]:
drawing(u10[t], v10[t], t=time[t])